In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

/home/mileva/mambaforge/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mileva/mambaforge/envs/torch/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mileva/Documents/Lyle/biked-commons/src/biked_commons/design_evaluation/../../biked_commons/prediction/usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malic

In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [4]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [5]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [6]:
eval_scores = evaluator(data_tens, condition)

/home/mileva/mambaforge/envs/torch/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [7]:
#check gradient of eval scores wrt data_tens
data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
eval_scores_sum = eval_scores.sum()
eval_scores_sum.backward()
print(data_tens.grad.shape)
print(data_tens.grad[0])



torch.Size([100, 94])
tensor([-2.4007e+00,  2.1912e+00,  1.6694e+00,  7.7458e+00, -3.6973e+00,
         7.3156e-01, -7.3366e-01, -5.3824e+00, -2.4013e+00, -9.5239e-01,
        -1.7396e-01, -3.1535e-01, -5.8118e-01, -1.3241e-01,  2.2539e-01,
        -5.6002e-02,  2.3076e-02,  1.0031e+00,  1.2599e-02,  3.6240e+00,
        -3.8225e-03,  7.3053e-02,  3.7936e-02,  3.0877e-03, -1.1037e-01,
        -1.0603e-02, -7.1568e-01, -6.7448e-01, -1.3151e-02, -2.2444e+00,
        -1.8268e+00,  3.8245e-02, -2.4888e-01,  0.0000e+00,  5.0000e-01,
         0.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         5.8358e-02, -5.3157e-02,  0.0000e+00, -9.6850e-02,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  3.5834e+00,
         0.0000e+00, -1.0000e+00,  0.0000e+00, -3.9906e+00,  1.4498e+00,
         0.0000e+00,  0.0000e

/home/mileva/mambaforge/envs/torch/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [8]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [9]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

In [10]:
main_scorer(data_tens.detach(), condition)

/home/mileva/mambaforge/envs/torch/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Hypervolume                     0.000000
Constraint Satisfaction Rate    0.841538
Maximum Mean Discrepancy        0.004870
dtype: float64

In [11]:
detailed_scorer(data_tens.detach(), condition)

/home/mileva/mambaforge/envs/torch/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Min Objective Score: Usability Score - 0 to 1                                                                 0.791411
Min Objective Score: Drag Force                                                                              27.933560
Min Objective Score: Knee Angle Error                                                                       186.338130
Min Objective Score: Hip Angle Error                                                                        822.850400
Min Objective Score: Arm Angle Error                                                                        861.650940
Min Objective Score: Mass                                                                                    22.190498
Min Objective Score: Planar Compliance                                                                      180.347920
Min Objective Score: Transverse Compliance                                                                  265.026400
Min Objective Score: Eccentric Compliance       